<a href="https://colab.research.google.com/github/pablonvsx/pisi3-ufrpe/blob/main/data-science/notebooks/ML/analise_final/Random_Forest_com_novo_r%C3%B3tulo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Experimento de Classificação com Random Forest

## Objetivo

Após a análise temporal dos dados e a reformulação do rótulo de qualidade da água, será realizado um novo conjunto de experimentos utilizando o algoritmo **Random Forest**. O objetivo desta etapa é avaliar a capacidade do modelo em classificar as condições ambientais dos corpos hídricos a partir das variáveis disponíveis para treinamento, considerando o novo cenário definido durante as análises anteriores.

Para reduzir os efeitos do desbalanceamento temporal identificado no conjunto de dados original, os experimentos serão conduzidos utilizando apenas o intervalo compreendido entre os anos **2000 e 2008**, período que apresentou maior representatividade das classes minoritárias e melhor distribuição dos registros de qualidade da água.

Além disso, será utilizada a nova estrutura de classificação composta por três categorias:

```text
Score 5 → Adequada
Score 4 → Boa
Score 0, 1, 2 ou 3 → Não adequada
```

A adoção desse novo rótulo foi motivada pelas dificuldades observadas na separação das antigas classes **Atenção** e **Crítica**, que apresentavam elevada sobreposição e geravam inconsistências nos resultados dos modelos. Ao agrupar essas categorias em uma única classe denominada **Não adequada**, busca-se representar de forma mais coerente situações que demandam atenção e possível validação técnica, sem exigir que o algoritmo realize distinções que os próprios dados demonstraram não sustentar adequadamente.

## Por que utilizar Random Forest?

O Random Forest foi selecionado por ser um algoritmo amplamente utilizado em problemas de classificação supervisionada, especialmente em cenários com dados heterogêneos, relações não lineares e presença de variáveis categóricas e numéricas.

Entre suas principais características estão:

* Construção de múltiplas árvores de decisão independentes;
* Redução da variância por meio do mecanismo de votação entre árvores;
* Menor sensibilidade a ruídos e outliers;
* Capacidade de capturar relações complexas entre variáveis;
* Menor propensão ao overfitting quando comparado a árvores individuais.

Além disso, o Random Forest foi um dos modelos que apresentou maior estabilidade durante os experimentos preliminares, tornando-se uma alternativa relevante para avaliar o impacto das alterações realizadas no conjunto de dados e na estrutura do rótulo.

## Estratégia Experimental

Os experimentos serão conduzidos utilizando a mesma configuração metodológica adotada nas etapas anteriores, incluindo:

* Divisão estratificada dos dados em treino e teste;
* Pré-processamento das variáveis categóricas por meio de One-Hot Encoding;
* Utilização exclusiva das variáveis de entrada selecionadas para evitar problemas de *data leakage*;
* Avaliação por meio das métricas de acurácia, precisão, recall e F1-score;
* Análise das matrizes de confusão para identificação dos principais padrões de erro.

Também serão avaliadas diferentes estratégias de balanceamento das classes, permitindo analisar como o algoritmo responde a diferentes distribuições de pesos e verificar possíveis melhorias no desempenho da categoria **Não adequada**.

## Expectativa

Espera-se que a combinação do novo recorte temporal com a reformulação do rótulo reduza os problemas de sobreposição observados nos experimentos anteriores, permitindo ao Random Forest construir fronteiras de decisão mais consistentes entre as três categorias propostas.

Os resultados obtidos nesta etapa serão posteriormente comparados com aqueles produzidos pelo LightGBM, permitindo identificar qual algoritmo apresenta melhor desempenho para o contexto do AquaSense e para o objetivo de apoiar a triagem inicial da qualidade da água.


## Preparação do ambiente


In [1]:
# IMPORTAÇÃO DE BIBLIOTECAS
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import os
from pathlib import Path
import warnings

warnings.filterwarnings("ignore")

SEED = 42

In [2]:
# DETECÇÃO DE AMBIENTE
try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# CONFIGURAÇÃO DE CAMINHO E CARREGAMENTO
if IN_COLAB:
    print("Ambiente Google Colab detectado.")

    drive.mount('/content/drive')

    DATA_PATH = Path(
        "/content/drive/MyDrive/EDA_AquaSense/Dataset/processed/water_quality_2000_2008_novorotulo.parquet"
    )

else:
    print("Ambiente local/VS Code detectado.")

    DATA_PATH = Path("../../dataset/water_quality_2000_2008_novorotulo.parquet")

# LEITURA DO DATASET
df = pd.read_parquet(DATA_PATH)

print("Dataset Parquet carregado com sucesso.")
print(f"Shape do dataset: {df.shape}")

df.head()

Ambiente Google Colab detectado.
Mounted at /content/drive
Dataset Parquet carregado com sucesso.
Shape do dataset: (59896, 23)


,Country,Area,Waterbody Type,Date,Ammonia (mg/l),Biochemical Oxygen Demand (mg/l),Dissolved Oxygen (mg/l),Orthophosphate (mg/l),pH (ph units),Temperature (cel),...,CCME_WQI,ph_ok,od_ok,dbo_ok,nitrate_ok,ammonia_limit,ammonia_ok,environmental_score,conama_status,Year
0,Canada,FISW_32,Lake,2003-12-02,0.043792,2.13333,9.824,0.00200,7.7900,12.00000,...,Excellent,1,1,1,1,2.0,1,5,Adequada,2003
1,Canada,IEEA_10_32,Lake,2001-06-08,0.015920,0.55000,9.824,0.00400,7.7900,16.80000,...,Excellent,1,1,1,1,2.0,1,5,Adequada,2001
2,Canada,CHRW-1876,River,2000-01-12,0.064400,10.87500,11.250,0.03590,8.2833,12.76150,...,Good,1,1,0,1,1.0,1,4,Boa,2000
3,Canada,ES063ESPFAA0000714,River,2004-01-12,1.071725,1.24444,5.850,0.20425,7.1000,18.32500,...,Fair,1,1,1,0,3.7,1,4,Boa,2004
4,Canada,CZPLA_391,River,2003-01-12,0.039740,1.83333,11.050,0.06100,7.7500,8.66667,...,Good,1,1,1,0,2.0,1,4,Boa,2003


In [29]:
df.columns.tolist()

['Country',
 'Area',
 'Waterbody Type',
 'Date',
 'Ammonia (mg/l)',
 'Biochemical Oxygen Demand (mg/l)',
 'Dissolved Oxygen (mg/l)',
 'Orthophosphate (mg/l)',
 'pH (ph units)',
 'Temperature (cel)',
 'Nitrogen (mg/l)',
 'Nitrate (mg/l)',
 'CCME_Values',
 'CCME_WQI',
 'ph_ok',
 'od_ok',
 'dbo_ok',
 'nitrate_ok',
 'ammonia_limit',
 'ammonia_ok',
 'environmental_score',
 'conama_status',
 'Year']

In [3]:
# preparando o ambiente para machine learning
import pandas as pd

from sklearn.model_selection import train_test_split

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

from sklearn.pipeline import Pipeline

from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score
)

In [4]:
# definição de x e y
X = df[
    [
        "Temperature (cel)",
        "Orthophosphate (mg/l)",
        "Country",
        "Waterbody Type",
        "Nitrogen (mg/l)"

    ]
]

y = df["conama_status"]

In [5]:
#train_test split

SEED = 42

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=SEED,
    stratify=y
)

print("Treino:", X_train.shape)
print("Teste:", X_test.shape)


Treino: (47916, 5)
Teste: (11980, 5)


In [6]:
# pré-processamento

categorical_features = [
    "Country",
    "Waterbody Type"

]

preprocessor = ColumnTransformer(
    transformers=[
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features
        )
    ],
    remainder="passthrough"
)

In [7]:
class_weights = {
    "Adequada": 1,
    "Boa": 2,
    "Não adequada": 2
}

In [14]:
model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),

        (
            "classifier",
            RandomForestClassifier(
                random_state=SEED,
                class_weight="balanced"
            )
        )
    ]
)

In [19]:
# sem balanceamento
model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),

        (
            "classifier",
            RandomForestClassifier(
                random_state=SEED
            )
        )
    ]
)

In [8]:
# com balanceamento manual
model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),

        (
            "classifier",
            RandomForestClassifier(
                random_state=SEED,
                class_weight=class_weights
            )
        )
    ]
)

In [20]:
model.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['Country',
                                                   'Waterbody Type'])])),
                ('classifier', RandomForestClassifier(random_state=42))])

## Avaliação das Métricas de Treino

Antes de analisar os resultados do conjunto de teste, também foi realizada a avaliação das métricas no conjunto de treino.

Essa etapa é importante porque permite comparar o comportamento do modelo entre treino e teste, ajudando na identificação de possíveis sinais de overfitting.

Por esse motivo, não basta apenas observar uma alta acurácia no treino. O ideal é que os resultados de treino e teste permaneçam relativamente próximos, indicando que o modelo conseguiu aprender padrões mais generalizáveis ao invés de apenas memorizar os dados.

Além da acurácia, também foram analisadas métricas como precision, recall e F1-score, permitindo observar como o modelo se comporta em cada classe individualmente.

In [16]:
y_train_pred = model.predict(X_train)

In [ ]:
trains_accuracy = accuracy_score(y_train, y_train_pred)

print("Train Accuracy:")
print(trains_accuracy)

Train Accuracy:
0.9047640095828287


In [11]:
# com balanceamento manual
train_accuracy = accuracy_score(y_train, y_train_pred)

train_precision = precision_score(
    y_train,
    y_train_pred,
    average='weighted'
)

train_recall = recall_score(
    y_train,
    y_train_pred,
    average='weighted'
)

train_f1 = f1_score(
    y_train,
    y_train_pred,
    average='weighted'
)

train_confusion_matrix = confusion_matrix(
    y_train,
    y_train_pred
)

print("Train Accuracy:")
print(train_accuracy)

print("Train Precision:")
print(train_precision)

print("Train Recall:")
print(train_recall)

print("Train F1:")
print(train_f1)

print("Classificator Report:")
print(classification_report(y_train, y_train_pred))
print("Train Confusion Matrix:")
print(train_confusion_matrix)

Train Accuracy:
0.9651264713248184
Train Precision:
0.9668908615512872
Train Recall:
0.9651264713248184
Train F1:
0.9655463231329471
Classificator Report:
              precision    recall  f1-score   support

    Adequada       0.99      0.98      0.98     32935
         Boa       0.96      0.93      0.94      9436
Não adequada       0.85      0.96      0.91      5545

    accuracy                           0.97     47916
   macro avg       0.94      0.96      0.94     47916
weighted avg       0.97      0.97      0.97     47916

Train Confusion Matrix:
[[32158   257   520]
 [  301  8747   388]
 [  132    73  5340]]


In [17]:
# com balanceamento automático
train_accuracy = accuracy_score(y_train, y_train_pred)

train_precision = precision_score(
    y_train,
    y_train_pred,
    average='weighted'
)

train_recall = recall_score(
    y_train,
    y_train_pred,
    average='weighted'
)

train_f1 = f1_score(
    y_train,
    y_train_pred,
    average='weighted'
)

train_confusion_matrix = confusion_matrix(
    y_train,
    y_train_pred
)

print("Train Accuracy:")
print(train_accuracy)

print("Train Precision:")
print(train_precision)

print("Train Recall:")
print(train_recall)

print("Train F1:")
print(train_f1)

print("Classificator Report:")
print(classification_report(y_train, y_train_pred))
print("Train Confusion Matrix:")
print(train_confusion_matrix)

Train Accuracy:
0.961453376742633
Train Precision:
0.9651906630649344
Train Recall:
0.961453376742633
Train F1:
0.9623420248443784
Classificator Report:
              precision    recall  f1-score   support

    Adequada       0.99      0.97      0.98     32935
         Boa       0.96      0.93      0.94      9436
Não adequada       0.82      0.98      0.89      5545

    accuracy                           0.96     47916
   macro avg       0.92      0.96      0.94     47916
weighted avg       0.97      0.96      0.96     47916

Train Confusion Matrix:
[[31854   347   734]
 [  194  8778   464]
 [   56    52  5437]]


In [21]:
# sem balanceamento
train_accuracy = accuracy_score(y_train, y_train_pred)

train_precision = precision_score(
    y_train,
    y_train_pred,
    average='weighted'
)

train_recall = recall_score(
    y_train,
    y_train_pred,
    average='weighted'
)

train_f1 = f1_score(
    y_train,
    y_train_pred,
    average='weighted'
)

train_confusion_matrix = confusion_matrix(
    y_train,
    y_train_pred
)

print("Train Accuracy:")
print(train_accuracy)

print("Train Precision:")
print(train_precision)

print("Train Recall:")
print(train_recall)

print("Train F1:")
print(train_f1)

print("Classificator Report:")
print(classification_report(y_train, y_train_pred))
print("Train Confusion Matrix:")
print(train_confusion_matrix)

Train Accuracy:
0.961453376742633
Train Precision:
0.9651906630649344
Train Recall:
0.961453376742633
Train F1:
0.9623420248443784
Classificator Report:
              precision    recall  f1-score   support

    Adequada       0.99      0.97      0.98     32935
         Boa       0.96      0.93      0.94      9436
Não adequada       0.82      0.98      0.89      5545

    accuracy                           0.96     47916
   macro avg       0.92      0.96      0.94     47916
weighted avg       0.97      0.96      0.96     47916

Train Confusion Matrix:
[[31854   347   734]
 [  194  8778   464]
 [   56    52  5437]]


In [12]:
y_pred = model.predict(X_test)

In [13]:
# com balanceamento manual
print("Accuracy:")
print(accuracy_score(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Accuracy:
0.7169449081803005

Classification Report:
              precision    recall  f1-score   support

    Adequada       0.82      0.87      0.85      8234
         Boa       0.47      0.37      0.42      2360
Não adequada       0.39      0.40      0.40      1386

    accuracy                           0.72     11980
   macro avg       0.56      0.55      0.55     11980
weighted avg       0.70      0.72      0.71     11980


Confusion Matrix:
[[7152  628  454]
 [1067  882  411]
 [ 460  371  555]]


In [18]:
# com balanceamento automático
print("Accuracy:")
print(accuracy_score(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Accuracy:
0.7169449081803005

Classification Report:
              precision    recall  f1-score   support

    Adequada       0.82      0.87      0.85      8234
         Boa       0.47      0.37      0.42      2360
Não adequada       0.39      0.40      0.40      1386

    accuracy                           0.72     11980
   macro avg       0.56      0.55      0.55     11980
weighted avg       0.70      0.72      0.71     11980


Confusion Matrix:
[[7152  628  454]
 [1067  882  411]
 [ 460  371  555]]


In [22]:
# sem balanceamento
print("Accuracy:")
print(accuracy_score(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Accuracy:
0.7169449081803005

Classification Report:
              precision    recall  f1-score   support

    Adequada       0.82      0.87      0.85      8234
         Boa       0.47      0.37      0.42      2360
Não adequada       0.39      0.40      0.40      1386

    accuracy                           0.72     11980
   macro avg       0.56      0.55      0.55     11980
weighted avg       0.70      0.72      0.71     11980


Confusion Matrix:
[[7152  628  454]
 [1067  882  411]
 [ 460  371  555]]


# Resultados do Random Forest com o Novo Rótulo

Após a reformulação do rótulo em três classes e a definição do recorte temporal entre os anos **2000 e 2008**, foi realizada uma nova bateria de experimentos utilizando o algoritmo **Random Forest**. O objetivo desta etapa foi avaliar o comportamento do modelo diante da nova estrutura de classificação e comparar seu desempenho com os resultados obtidos anteriormente pelo LightGBM.

A nova configuração do problema considera as seguintes categorias:

```text
Score 5 → Adequada
Score 4 → Boa
Score 0, 1, 2 ou 3 → Não adequada
```

Essa reformulação teve como principal objetivo reduzir a sobreposição observada entre as antigas classes **Atenção** e **Crítica**, que representavam a principal fonte de erro dos modelos durante os experimentos iniciais.

## Estratégias avaliadas

Foram testadas três abordagens distintas:

* Random Forest sem balanceamento;
* Random Forest com balanceamento automático;
* Random Forest com balanceamento manual.

A estratégia de balanceamento manual utilizou os seguintes pesos:

```text
Adequada = 1
Boa = 1
Não adequada = 2
```

## Análise dos Resultados

Os resultados obtidos mostraram um comportamento bastante interessante. Independentemente da estratégia de balanceamento utilizada, o desempenho final do modelo no conjunto de teste permaneceu praticamente inalterado.

A acurácia observada foi de aproximadamente:

```text
72%
```

em todas as configurações avaliadas.

Além disso, as métricas das três classes também permaneceram extremamente próximas entre os diferentes cenários, indicando que o algoritmo já estava aprendendo adequadamente os padrões presentes no conjunto de dados sem depender fortemente de ajustes de pesos.

### Classe Adequada

A classe **Adequada** apresentou o melhor desempenho entre todas as categorias avaliadas.

No conjunto de teste, os resultados ficaram próximos de:

```text
Precisão = 82%
Recall = 87%
F1-score = 85%
```

Esses valores demonstram que o modelo consegue identificar com elevada confiabilidade os corpos hídricos que apresentam melhores condições ambientais.

### Classe Boa

A classe **Boa** apresentou um desempenho intermediário.

No conjunto de teste, foram observados aproximadamente:

```text
Precisão = 47%
Recall = 37%
F1-score = 42%
```

Apesar de inferiores aos resultados da classe Adequada, esses valores indicam que o modelo consegue capturar parte dos padrões associados a essa categoria, embora ainda existam dificuldades na separação entre situações de qualidade intermediária.

### Classe Não Adequada

A classe **Não adequada**, que reúne os antigos casos classificados como Atenção e Crítica, apresentou os seguintes resultados:

```text
Precisão = 39%
Recall = 40%
F1-score = 40%
```

Embora essas métricas sejam inferiores às observadas para a classe Adequada, elas representam uma melhora considerável em relação aos experimentos realizados com quatro classes, nos quais a antiga categoria Crítica apresentava forte instabilidade e elevadas taxas de erro.

Além disso, observa-se um aspecto importante: a diferença entre precisão e recall permaneceu pequena. Isso indica que o modelo não está excessivamente enviesado para produzir falsos positivos ou falsos negativos, apresentando um comportamento mais equilibrado e consistente.

## Avaliação do Overfitting

Um dos pontos mais relevantes observados durante os experimentos foi a diferença significativa entre as métricas de treino e teste.

No conjunto de treino, o Random Forest atingiu acurácia superior a 96%, enquanto no conjunto de teste os resultados ficaram próximos de 72%.

Essa diferença sugere que o modelo possui uma tendência relativamente elevada ao overfitting, comportamento esperado em algoritmos baseados em múltiplas árvores de decisão quando não há um controle mais rigoroso de complexidade.

Apesar disso, o desempenho no conjunto de teste permaneceu estável entre os diferentes experimentos realizados, indicando que o modelo ainda consegue generalizar parte dos padrões presentes nos dados.

## Comparação entre Estratégias de Balanceamento

Um resultado particularmente interessante foi a baixa influência do balanceamento sobre o desempenho final do modelo.

Diferentemente do que ocorreu em experimentos anteriores utilizando quatro classes, onde o balanceamento alterava significativamente a relação entre precisão e recall, no novo cenário de três classes o Random Forest apresentou resultados praticamente idênticos com:

* Balanceamento manual;
* Balanceamento automático;
* Ausência de balanceamento.

Esse comportamento sugere que a reformulação do rótulo teve um impacto muito maior sobre o desempenho do modelo do que as estratégias de balanceamento propriamente ditas.

## Conclusão

Os resultados obtidos indicam que a principal melhoria não veio do ajuste dos pesos das classes, mas sim da redefinição do problema de classificação.

Ao agrupar as antigas categorias **Atenção** e **Crítica** em uma única classe denominada **Não adequada**, foi possível reduzir a sobreposição entre classes e construir um problema mais coerente com a estrutura dos dados disponíveis.

O Random Forest apresentou desempenho estável em todos os cenários avaliados, com métricas equilibradas para a classe **Não adequada** e boa capacidade de identificação das amostras classificadas como **Adequada**.

Esses resultados reforçam a hipótese levantada durante as análises exploratórias: a principal limitação do problema estava associada à definição do rótulo e à elevada similaridade entre as antigas classes Atenção e Crítica, e não necessariamente à escolha do algoritmo ou à estratégia de balanceamento utilizada.
